In [1]:
import sys, os
import random
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#from src.data.data_collector import DataCollector
from src.models.model_trainer_rl_v2_2_buyhold import ModelTrainerRL, TradingEnvRL
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from src.models.backtester import PortfolioBacktester, PortfolioBacktesterRL
from src.utils.config_loader import load_config


config = load_config("config/config.yaml")

DEFAULT_SEED = 42

def set_global_seed(seed=DEFAULT_SEED):
    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    return seed

d:\MSDS\buy-sell-hold-strategy-prediction\venv-3.11\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
d:\MSDS\buy-sell-hold-strategy-prediction\venv-3.11\Lib\site-packages\kaleido\scopes\plotly.py:32: DeprecationWarning:


Use of plotly.io.kaleido.scope.default_format is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_format instead.


d:\MSDS\buy-sell-hold-strategy-prediction\venv-3.11\Lib\site-packages\kaleido\scopes\plotly.py:33: DeprecationWarning:




## V2.2 SAC Agent

### Function

In [2]:
def run_sac_trading_pipeline(
    stock_symbol,
    config,
    year='2022',
    save_path="models/",
    show_plot=True,
    seed=42,
    n_eval_episodes=10,
):

    """
    Complete SAC trading pipeline: load data, train model, generate predictions, and backtest.

    Parameters:
    -----------
    stock_symbol : str
        Stock ticker symbol (e.g., 'CNP', 'MDU')
    config : dict
        Configuration dictionary loaded from config.yaml
    year : str, optional
        Year for data file selection (default: '2022')
    save_path : str, optional
        Directory to save/load models (default: 'models/')
    show_plot : bool, optional
        Whether to display the portfolio plot (default: True)
    seed : int, optional
        Fixed random seed for repeatable runs (default: 42)
    n_eval_episodes : int, optional
        Number of evaluation episodes for behavior metrics (default: 10)

    Returns:
    --------
    tuple : (portfolio, metrics, actions)
        - portfolio: VectorBT portfolio object
        - metrics: Dictionary of performance metrics
        - actions: Array of predicted actions
    """
    import inspect
    import random

    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

    print(f"Using fixed seed: {seed}")

    # 1. Load Data
    try:
        data = pd.read_csv(f"data/processed/{stock_symbol}_processed_{year}.csv")
        if "Date" in data.columns:
            data["Date"] = pd.to_datetime(data["Date"])
            data.set_index("Date", inplace=True)
        print(f"Data loaded successfully for {stock_symbol}.")
    except FileNotFoundError:
        print(f"Error: Data file not found for {stock_symbol}. Check path.")
        sys.exit()

    # 2. Split Train/Test
    split_idx = int(len(data) * 0.7)
    train_df = data.iloc[:split_idx]
    test_df = data.iloc[split_idx:]

    # 3. Training Phase
    print(f"Training SAC Agent for {stock_symbol}...")
    trainer = ModelTrainerRL(config["reinforcement_learning"])
    env_params = config["reinforcement_learning"]["environment"]

    env_train = TradingEnvRL(
        train_df,
        initial_balance=env_params.get("initial_balance", 10000),
        commission=env_params.get("commission", 0.001),
        lookback_window=env_params.get("lookback_window", 30),
        reward_func="profit",
    )

    result = trainer.train_sac(env_train)
    trained_model = result["model"]
    print(f"Training result keys: {list(result.keys())}")
    print(f"Trained model type: {type(trained_model).__name__}")
    if "vec_env" in result:
        print(f"Vectorized env type: {type(result['vec_env']).__name__}")
    trainer.save_models(save_path)
    print("Training complete. Models saved.")

    # Extract SAC training diagnostics if available
    sac_diag = {}
    logger_values = getattr(getattr(trained_model, "logger", None), "name_to_value", {})
    for key in [
        "train/entropy_loss",
        "train/policy_gradient_loss",
        "train/qf1_loss",
        "train/qf2_loss",
        "train/actor_loss",
        "train/critic_loss",
        "train/ent_coef",
    ]:
        val = logger_values.get(key, None) if isinstance(logger_values, dict) else None
        if val is not None:
            sac_diag[key] = float(val)

    # 4. Inference Phase
    print("Generating Agent Predictions on Test Data...")
    model = SAC.load(os.path.join(save_path, "sac_model"))

    env_test = TradingEnvRL(
        test_df,
        initial_balance=env_params.get("initial_balance", 100000),
        commission=env_params.get("commission", 0.001),
        lookback_window=env_params.get("lookback_window", 30),
        reward_func="profit",
    )

    vec_env_test = DummyVecEnv([lambda: env_test])

    norm_path = os.path.join(save_path, "sac_vecnormalize.pkl")
    if os.path.exists(norm_path):
        vec_env_test = VecNormalize.load(norm_path, vec_env_test)
        vec_env_test.training = False
        vec_env_test.norm_reward = False
    else:
        print("WARNING: Normalization stats not found. Model predictions may be garbage.")

    if hasattr(vec_env_test, "seed"):
        vec_env_test.seed(seed)

    obs = vec_env_test.reset()
    done = [False]
    actions = []

    while not done[0]:
        action, _ = model.predict(obs, deterministic=True)
        actions.append(float(action[0]))
        obs, _, done, _ = vec_env_test.step(action)

    print(f"Generated {len(actions)} actions.")

    # Evaluate behavior over multiple episodes
    eval_rewards = []
    eval_lengths = []
    eval_returns = []

    for ep in range(max(1, int(n_eval_episodes))):
        np.random.seed(seed + ep)
        random.seed(seed + ep)
        try:
            import torch
            torch.manual_seed(seed + ep)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed + ep)
        except ImportError:
            pass

        obs_ep, _ = env_test.reset(seed=seed + ep)
        done_ep = False
        ep_reward = 0.0
        ep_len = 0
        last_info = {}

        while not done_ep:
            action_ep, _ = model.predict(obs_ep, deterministic=True)
            obs_ep, reward_ep, terminated_ep, truncated_ep, info_ep = env_test.step(action_ep)
            done_ep = terminated_ep or truncated_ep
            ep_reward += float(reward_ep)
            ep_len += 1
            last_info = info_ep

        eval_rewards.append(ep_reward)
        eval_lengths.append(ep_len)
        eval_returns.append(float(last_info.get("return", 0.0)))

    metric_kwargs = dict(
        episode_rewards=eval_rewards,
        episode_lengths=eval_lengths,
        episode_returns=eval_returns,
        success_threshold=0.0,
    )
    if "random_state" in inspect.signature(trainer.calculate_rl_performance_metrics).parameters:
        metric_kwargs["random_state"] = seed

    rl_metrics = trainer.calculate_rl_performance_metrics(**metric_kwargs)

    actions_arr = np.array(actions).flatten()
    eps = 1e-6
    action_low_boundary_pct = float(np.mean(actions_arr <= (0.0 + eps)) * 100.0) if len(actions_arr) > 0 else 0.0
    action_high_boundary_pct = float(np.mean(actions_arr >= (1.0 - eps)) * 100.0) if len(actions_arr) > 0 else 0.0

    # 5. Backtesting Phase
    print("Running Backtest...")
    backtester = PortfolioBacktesterRL(env_params)

    portfolio = backtester.run_backtest(
        price_data=test_df["close"],
        predicted_weights=np.array(actions).flatten(),
        lookback_window=env_params.get("lookback_window", 30),
    )

    comparison = backtester.compare_with_buy_and_hold_rl()
    metrics = backtester.get_performance_metrics()

    print(f"\n--- Strategy Performance for {stock_symbol} ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    print("\n--- Result Summary ---")
    print(f"Seed: {seed}")
    print(f"Actions generated: {len(actions)}")
    if isinstance(comparison, dict):
        print("Buy & Hold comparison:")
        for k, v in comparison.items():
            print(f"{k}: {v}")
    else:
        print(f"Buy & Hold comparison: {comparison}")

    print("\n--- Update Speed Metrics (New Incoming Data Adaptation) ---")
    print(f"Learning improvement: {rl_metrics.get('learning_improvement', 0.0):.6f}")
    print(f"Learning rate pct: {rl_metrics.get('learning_rate_pct', 0.0):.4f}%")
    print(f"Trend slope: {rl_metrics.get('trend_slope', 0.0):.6f}")

    print("\n--- Stability / Random Behavior Metrics ---")
    print(f"Mean episode reward: {rl_metrics.get('mean_episode_reward', 0.0):.6f}")
    print(f"Std episode reward: {rl_metrics.get('std_episode_reward', 0.0):.6f}")
    print(f"Coefficient of variation: {rl_metrics.get('coefficient_of_variation', 0.0):.4f}%")
    print(f"Variance ratio late/early: {rl_metrics.get('variance_ratio_late_vs_early', 0.0):.6f}")
    print(f"Success rate: {rl_metrics.get('success_rate_pct', 0.0):.2f}%")
    print(f"Win rate: {rl_metrics.get('win_rate_pct', 0.0):.2f}%")

    print("\n--- Divergence / Failure-Case Features ---")
    print(f"Worst episode idx: {rl_metrics.get('worst_episode_idx', -1)}")
    print(f"Worst episode reward: {rl_metrics.get('worst_episode_reward', 0.0):.6f}")
    print(f"Worst episode return: {rl_metrics.get('worst_episode_return', 0.0):.6f}")
    print(f"Max drawdown pct: {rl_metrics.get('max_drawdown_pct', 0.0):.4f}%")
    negative_returns = int(np.sum(np.array(eval_returns) < 0.0))
    print(f"Negative-return episodes: {negative_returns}/{len(eval_returns)}")

    print("\n--- SAC Diagnostics ---")
    if len(sac_diag) > 0:
        for key, value in sac_diag.items():
            print(f"{key}: {value:.6f}")
    else:
        print("SAC training diagnostics are not available from logger in this run.")
    print(f"Action boundary at 0.0 (%): {action_low_boundary_pct:.2f}%")
    print(f"Action boundary at 1.0 (%): {action_high_boundary_pct:.2f}%")

    if show_plot:
        portfolio.plot().show()
        trades = portfolio.trades.records_readable
        print(f"\n--- Trade Statistics for {stock_symbol} ---")
        print(f"Total number of trades: {len(trades)}")

        if "PnL" in trades.columns:
            profitable_trades = trades[trades["PnL"] > 0]
            loss_trades = trades[trades["PnL"] < 0]

            print(f"\n--- Trade Outcomes ---")
            print(f"Number of profitable exits: {len(profitable_trades)}")
            print(f"Number of loss exits (cut loss): {len(loss_trades)}")
            print(f"Win rate: {len(profitable_trades) / len(trades) * 100:.2f}%")
            print(
                f"Average profit per winning trade: ${profitable_trades['PnL'].mean():.2f}"
                if len(profitable_trades) > 0 else "No profitable trades"
            )
            print(
                f"Average loss per losing trade: ${loss_trades['PnL'].mean():.2f}"
                if len(loss_trades) > 0 else "No losing trades"
            )

    return portfolio, metrics, np.array(actions).flatten()

### RUN

In [ ]:
# Single stock quick demo run (reduced timesteps for fast output preview)
import copy

quick_config = copy.deepcopy(config)
quick_config["reinforcement_learning"]["sac"]["total_timesteps"] = 200000

stock_symbol = "AAPL"
portfolio, metrics, actions = run_sac_trading_pipeline(
    stock_symbol=stock_symbol,
    config=quick_config,
    show_plot=True,
    seed=42,
    n_eval_episodes=10,
    save_path="models/quick_demo",
)

In [3]:
# ## Single stock
# stock_symbol = "CNP"
# portfolio, metrics, actions = run_ppo_trading_pipeline(stock_symbol, config)

In [4]:
# # portfolio.trades.records_readable
# # Print trade statistics
# trades = portfolio.trades.records_readable
# print(f"\n--- Trade Statistics for abc ---")
# print(f"Total number of trades: {len(trades)}")
# print("\nTrade Direction Counts:")
# print(trades['Direction'].value_counts())

# # Analyze trade outcomes
# if 'PnL' in trades.columns:
#         profitable_trades = trades[trades['PnL'] > 0]
#         loss_trades = trades[trades['PnL'] < 0]
        
#         print(f"\n--- Trade Outcomes ---")
#         print(f"Number of profitable exits: {len(profitable_trades)}")
#         print(f"Number of loss exits (cut loss): {len(loss_trades)}")
#         print(f"Win rate: {len(profitable_trades) / len(trades) * 100:.2f}%")
#         print(f"Average profit per winning trade: ${profitable_trades['PnL'].mean():.2f}" if len(profitable_trades) > 0 else "No profitable trades")
#         print(f"Average loss per losing trade: ${loss_trades['PnL'].mean():.2f}" if len(loss_trades) > 0 else "No losing trades")
    


In [3]:
# Multiple stocks in a loop
stocks = ["AAPL", "AMZN", "TSLA", "BAC","MDU", "CWCO", "NEE", "DUK"]
results = {}

for stock in stocks:
    print(f"\n{'='*60}")
    print(f"Processing {stock}")
    print(f"{'='*60}")
    portfolio, metrics, actions = run_sac_trading_pipeline(
        stock_symbol=stock, 
        config=config,
        seed=DEFAULT_SEED,
        n_eval_episodes=10,
        show_plot=True  # Don't show plots in loop
    )
    results[stock] = {'portfolio': portfolio, 'metrics': metrics, 'actions': actions}


Processing AAPL
Using fixed seed: 42
Data loaded successfully for AAPL.
Training SAC Agent for AAPL...
Using cpu device


C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_24088\162156532.py:56: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training SAC for 150000 timesteps


---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 50       |
|    time_elapsed    | 46       |
|    total_timesteps | 2348     |
| train/             |          |
|    actor_loss      | -5.18    |
|    critic_loss     | 0.0114   |
|    ent_coef        | 0.51     |
|    ent_coef_loss   | -1.12    |
|    learning_rate   | 0.0003   |
|    n_updates       | 2247     |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 49       |
|    time_elapsed    | 94       |
|    total_timesteps | 4696     |
| train/             |          |
|    actor_loss      | -6.89    |
|    critic_loss     | 0.00674  |
|    ent_coef        | 0.252    |
|    ent_coef_loss   | -2.28    |
|    learning_rate   | 0.0003   |
|    n_updates       | 4595     |
---------------------------------
---------------------------------
| time/       

C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_24088\162156532.py:137: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Generated 218 actions.


...

Generated 218 actions.


INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:RL AGENT PERFORMANCE METRICS SUMMARY
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:Total Episodes: 10
INFO:src.models.model_trainer_rl_v2_2_buyhold:Mean Episode Reward: 15.88 ± 0.00
INFO:src.models.model_trainer_rl_v2_2_buyhold:Success Rate: 100.0%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Learning Improvement: 0.00 (0.0%)
INFO:src.models.model_trainer_rl_v2_2_buyhold:Trend Slope: 0.0000
INFO:src.models.model_trainer_rl_v2_2_buyhold:Stability (CV): 0.00%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Portfolio Sharpe Ratio: 0.000
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:BacktesterRL:Preparing Backtest. Raw Prices: 279, Predictions: 218
INFO:BacktesterRL:Runn

Running Backtest...

--- Strategy Performance for DUK ---
Total Return (%): 31.9800
Annual Return (%): 59.1300
Sharpe Ratio: 4.2118
Sortino Ratio: 9.0018
Max Drawdown (%): -3.3900
Calmar Ratio: 17.4544
Win Rate (%): 75.2100
Total Trades: 121.0000
Final Value ($): 131979.5300

--- Result Summary ---
Seed: 42
Actions generated: 218
Buy & Hold comparison:                            RL Strategy  Buy & Hold
Date                                              
2025-01-10 00:00:00-05:00    -0.000926   -0.000999
2025-01-13 00:00:00-05:00     0.001238    0.001549
2025-01-14 00:00:00-05:00     0.005874    0.008248
2025-01-15 00:00:00-05:00     0.002172    0.004474
2025-01-16 00:00:00-05:00     0.026298    0.029102
...                                ...         ...
2025-11-14 00:00:00-05:00     0.328272    0.199784
2025-11-17 00:00:00-05:00     0.343574    0.223935
2025-11-18 00:00:00-05:00     0.332111    0.210442
2025-11-19 00:00:00-05:00     0.320160    0.195091
2025-11-20 00:00:00-05:00     0.3


--- Trade Statistics for DUK ---
Total number of trades: 121

--- Trade Outcomes ---
Number of profitable exits: 91
Number of loss exits (cut loss): 30
Win rate: 75.21%
Average profit per winning trade: $440.40
Average loss per losing trade: $-269.91


d:\MSDS\buy-sell-hold-strategy-prediction\venv-3.11\Lib\site-packages\vectorbt\records\base.py:624: FutureWarning:

Setting an Index with object dtype into a DataFrame will stop inferring another dtype in a future version. Cast the Index explicitly before setting it into the DataFrame.

